In [1]:
from scipy.interpolate import CubicSpline
from datetime import timedelta
from pathlib import Path
import polars as pl
import os
import pandas as pd
import pickle
from tqdm.notebook import tqdm

# Setup paths
processed_data_dir = Path(
    r"F:\PyTorch_GPU\maritime_monitoring_preprocessing\processed_data"
)
checkpoint_dir = Path(
    r"F:\PyTorch_GPU\maritime_monitoring_preprocessing\checkpoint_interpolated"
)
checkpoint_dir.mkdir(exist_ok=True, parents=True)

print(f"Input data directory: {processed_data_dir}")
print(f"Checkpoint directory: {checkpoint_dir}")
print(f"Directory exists: {processed_data_dir.exists()}")

# List available data files
if processed_data_dir.exists():
    files = (
        list(processed_data_dir.glob("*.csv"))
        + list(processed_data_dir.glob("*.parquet"))
        + list(processed_data_dir.glob("*.pkl"))
    )
    print(f"\nAvailable files ({len(files)}):")
    for f in sorted(files)[:10]:  # Show first 10
        print(f"  - {f.name}")

Input data directory: F:\PyTorch_GPU\maritime_monitoring_preprocessing\processed_data
Checkpoint directory: F:\PyTorch_GPU\maritime_monitoring_preprocessing\checkpoint_interpolated
Directory exists: True

Available files (8):
  - AIS_2020_01_05.csv
  - AIS_2020_01_06.csv
  - AIS_2020_01_07.csv
  - AIS_2020_01_08.csv
  - AIS_2020_01_09.csv
  - AIS_2020_01_10.csv
  - AIS_2020_01_11.csv
  - AIS_2020_01_12.csv


In [2]:
import numpy as np

# GPU acceleration support
NUMBA_GPU_AVAILABLE = False
try:
    from numba import cuda, jit

    NUMBA_GPU_AVAILABLE = cuda.is_available()
except:
    pass


@jit(nopython=False) if NUMBA_GPU_AVAILABLE else lambda f: f
def interpolate_vessel_trajectory(
    df_vessel,
    target_interval_minutes=5,
    time_col="timestamp",
    lat_col="lat",
    lon_col="lon",
):
    """
    Interpolate a single vessel's trajectory at regular 5-minute intervals using cubic spline.
    GPU-accelerated if CUDA available.

    Parameters:
    -----------
    df_vessel : pd.DataFrame
        Vessel trajectory data with time, lat, lon columns
    target_interval_minutes : int
        Target time interval for interpolation (default: 5 minutes)
    time_col, lat_col, lon_col : str
        Column names for time, latitude, longitude

    Returns:
    --------
    pd.DataFrame
        Interpolated trajectory with columns: ['timestamp', 'lat', 'lon', 'interpolated']
    """

    if len(df_vessel) < 2:
        return pd.DataFrame()

    # Select only required columns and drop NaN
    required_cols = [time_col, lat_col, lon_col]
    if not all(col in df_vessel.columns for col in required_cols):
        return pd.DataFrame()

    df = df_vessel[[time_col, lat_col, lon_col]].dropna().copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    if len(df) < 2:
        return pd.DataFrame()

    # Convert timestamps to numeric (seconds since start)
    t_start = df[time_col].min()
    t_numeric = (df[time_col] - t_start).dt.total_seconds().values

    lat_vals = df[lat_col].values
    lon_vals = df[lon_col].values

    # Build cubic splines for lat/lon
    try:
        cs_lat = CubicSpline(t_numeric, lat_vals, bc_type="not-a-knot")
        cs_lon = CubicSpline(t_numeric, lon_vals, bc_type="not-a-knot")
    except Exception as e:
        return pd.DataFrame()

    # Create new time grid at target_interval_minutes
    t_min, t_max = t_numeric.min(), t_numeric.max()
    dt_seconds = target_interval_minutes * 60
    t_interp = np.arange(t_min, t_max + dt_seconds, dt_seconds)

    # Evaluate splines at new time points
    lat_interp = cs_lat(t_interp)
    lon_interp = cs_lon(t_interp)

    # Convert back to timestamps
    timestamps_interp = [t_start + timedelta(seconds=float(t)) for t in t_interp]

    # Mark which points are original vs interpolated
    is_interpolated = ~np.isin(t_interp, t_numeric)

    result = pd.DataFrame(
        {
            "timestamp": timestamps_interp,
            "lat": lat_interp,
            "lon": lon_interp,
            "interpolated": is_interpolated,
        }
    )

    return result


print(f"✓ Interpolation function defined (GPU acceleration: {NUMBA_GPU_AVAILABLE})")

✓ Interpolation function defined (GPU acceleration: True)


In [3]:
def process_daywise_data(
    input_dir,
    checkpoint_dir,
    time_col="timestamp",
    lat_col="lat",
    lon_col="lon",
    mmsi_col="MMSI",
):
    """
    Process AIS data day-wise: load, interpolate, save, and collect statistics.

    Parameters:
    -----------
    input_dir : Path
        Directory containing daily AIS data files
    checkpoint_dir : Path
        Directory to save interpolated data
    time_col, lat_col, lon_col, mmsi_col : str
        Column names in the data

    Returns:
    --------
    dict
        Statistics including data points and vessel counts per day
    """

    stats = {}
    checkpoint_dir.mkdir(exist_ok=True, parents=True)

    # Get all data files
    csv_files = sorted(input_dir.glob("*.csv"))
    parquet_files = sorted(input_dir.glob("*.parquet"))
    all_files = csv_files + parquet_files

    print(f"Found {len(all_files)} files to process\n")
    print("=" * 100)
    print(
        f"{'Date':<12} {'File':<40} {'Total Points':<15} {'Unique Vessels':<15} {'Status':<20}"
    )
    print("=" * 100)

    for file_idx, file_path in enumerate(all_files, 1):
        try:
            # Extract date from filename
            date_str = file_path.stem.split("_")[-1]  # Assumes format: *_YYYY_MM_DD

            # Load data
            if file_path.suffix == ".csv":
                df = pd.read_csv(file_path)
            elif file_path.suffix == ".parquet":
                df = pd.read_parquet(file_path)
            else:
                continue

            if df.empty:
                print(
                    f"{date_str:<12} {file_path.name:<40} {'0':<15} {'0':<15} {'⚠️  Empty':<20}"
                )
                stats[date_str] = {
                    "total_points": 0,
                    "unique_vessels": 0,
                    "interpolated_points": 0,
                    "interpolated_vessels": 0,
                    "file": file_path.name,
                }
                continue

            # Convert timestamp column to datetime if needed
            if time_col in df.columns and df[time_col].dtype == "object":
                df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

            # Check for required columns
            required = [time_col, lat_col, lon_col, mmsi_col]
            if not all(col in df.columns for col in required):
                print(
                    f"{date_str:<12} {file_path.name:<40} {len(df):<15} {'?':<15} {'❌ Missing cols':<20}"
                )
                continue

            # Get statistics before interpolation
            total_points_before = len(df)
            unique_vessels = df[mmsi_col].nunique()

            # Interpolate per vessel
            interpolated_data = {}
            total_interp_points = 0

            for mmsi in df[mmsi_col].unique():
                vessel_df = df[df[mmsi_col] == mmsi].copy()
                interp_df = interpolate_vessel_trajectory(
                    vessel_df,
                    time_col=time_col,
                    lat_col=lat_col,
                    lon_col=lon_col,
                )

                if not interp_df.empty:
                    interpolated_data[mmsi] = interp_df
                    total_interp_points += len(interp_df)

            # Save interpolated data as pickle and parquet
            if interpolated_data:
                # Save as pickle for backwards compatibility
                pkl_file = checkpoint_dir / f"interpolated_{date_str}.pkl"
                with open(pkl_file, "wb") as f:
                    pickle.dump(interpolated_data, f)

                # Save as parquet (convert dict of DataFrames to single DataFrame with MMSI column)
                dfs_to_concat = []
                for mmsi, idf in interpolated_data.items():
                    idf_copy = idf.copy()
                    idf_copy["MMSI"] = mmsi
                    dfs_to_concat.append(idf_copy)

                if dfs_to_concat:
                    combined_df = pd.concat(dfs_to_concat, ignore_index=True)
                    parquet_file = checkpoint_dir / f"interpolated_{date_str}.parquet"
                    combined_df.to_parquet(parquet_file, compression="snappy")

                status = f"✓ Saved ({len(interpolated_data)} vessels)"
            else:
                status = "❌ No data interpolated"
                total_interp_points = 0

            # Store statistics
            stats[date_str] = {
                "total_points_before": total_points_before,
                "total_points_after": total_interp_points,
                "unique_vessels": unique_vessels,
                "vessels_interpolated": len(interpolated_data),
                "file": file_path.name,
            }

            print(
                f"{date_str:<12} {file_path.name:<40} {total_points_before:<15} {unique_vessels:<15} {status:<20}"
            )

        except Exception as e:
            print(
                f"{date_str:<12} {file_path.name:<40} {'?':<15} {'?':<15} {'❌ Error: ' + str(e)[:10]:<20}"
            )
            continue

    print("=" * 100)
    return stats


print("✓ Process function defined")

✓ Process function defined


In [4]:
def detect_column_names(df):
    """
    Auto-detect column names for time, latitude, longitude, and MMSI.

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe to detect column names from

    Returns:
    --------
    dict
        Dictionary with detected column names
    """
    columns = df.columns.tolist()

    # Detect time column
    time_col = None
    for col in ["BaseDateTime", "timestamp", "DateTime", "Time", "Timestamp"]:
        if col in columns:
            time_col = col
            break

    # Detect latitude column
    lat_col = None
    for col in ["LAT", "Latitude", "lat", "Lat"]:
        if col in columns:
            lat_col = col
            break

    # Detect longitude column
    lon_col = None
    for col in ["LON", "Longitude", "lon", "Long"]:
        if col in columns:
            lon_col = col
            break

    # Detect MMSI column
    mmsi_col = None
    for col in ["MMSI", "mmsi", "Mmsi"]:
        if col in columns:
            mmsi_col = col
            break

    # Detect IMO column (optional)
    imo_col = None
    for col in ["IMO", "imo", "Imo"]:
        if col in columns:
            imo_col = col
            break

    return {
        "time_col": time_col,
        "lat_col": lat_col,
        "lon_col": lon_col,
        "mmsi_col": mmsi_col,
        "imo_col": imo_col,
    }


print("✓ Column detection function defined")

✓ Column detection function defined


In [ ]:
import gc
import sys


def process_single_file(file_path, checkpoint_dir, detected_cols=None):
    """
    Process a single AIS data file: load, interpolate, save.
    Minimal memory footprint - returns immediately after saving.

    Parameters:
    -----------
    file_path : Path
        Path to the data file
    checkpoint_dir : Path
        Directory to save results
    detected_cols : dict
        Detected column names from first file

    Returns:
    --------
    tuple: (date_str, stats_dict, updated_detected_cols)
    """

    date_str = file_path.stem.split("_")[-1]

    try:
        # Load with pandas (simpler, more stable)
        encodings = ["utf-8", "latin1", "iso-8859-1", "cp1252"]
        df = None

        for encoding in encodings:
            try:
                df = pd.read_csv(
                    file_path,
                    on_bad_lines="skip",
                    encoding=encoding,
                    low_memory=False,
                )
                if df is not None and not df.empty:
                    break
            except:
                continue

        if df is None or df.empty:
            return date_str, {"error": "Failed to load file"}, detected_cols

        # Auto-detect columns if not provided
        if detected_cols is None:
            detected_cols = detect_column_names(df)

        time_col = detected_cols["time_col"]
        lat_col = detected_cols["lat_col"]
        lon_col = detected_cols["lon_col"]
        mmsi_col = detected_cols["mmsi_col"]
        imo_col = detected_cols["imo_col"]

        if not all([time_col, lat_col, lon_col, mmsi_col]):
            return (
                date_str,
                {"error": f"Missing cols: {[time_col, lat_col, lon_col, mmsi_col]}"},
                detected_cols,
            )

        # Convert timestamp
        if df[time_col].dtype == "object":
            df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

        total_points_before = len(df)
        unique_vessels = df[mmsi_col].nunique()

        # Process each vessel
        interpolated_data = {}
        total_points_after = 0

        for mmsi in tqdm(
            df[mmsi_col].unique(), desc=f"  {date_str}", leave=False, unit="v"
        ):
            try:
                vessel_df = df[df[mmsi_col] == mmsi].copy()
                interp_df = interpolate_vessel_trajectory(
                    vessel_df,
                    time_col=time_col,
                    lat_col=lat_col,
                    lon_col=lon_col,
                )

                if not interp_df.empty:
                    if imo_col and imo_col in vessel_df.columns:
                        interp_df["IMO"] = vessel_df[imo_col].iloc[0]

                    interpolated_data[mmsi] = interp_df
                    total_points_after += len(interp_df)

                del vessel_df
            except:
                pass

        # Save to checkpoint
        if interpolated_data:
            pkl_file = checkpoint_dir / f"interpolated_{date_str}.pkl"
            with open(pkl_file, "wb") as f:
                pickle.dump(interpolated_data, f, protocol=pickle.HIGHEST_PROTOCOL)

            # Save parquet
            dfs = []
            for mmsi, idf in interpolated_data.items():
                idf_copy = idf.copy()
                idf_copy["MMSI"] = mmsi
                dfs.append(idf_copy)

            if dfs:
                combined_df = pd.concat(dfs, ignore_index=True)
                parquet_file = checkpoint_dir / f"interpolated_{date_str}.parquet"
                combined_df.to_parquet(parquet_file, compression="snappy")
                del combined_df, dfs

        # Return stats with CORRECT KEY NAMES
        stats = {
            "total_points_before": total_points_before,
            "total_points_after": total_points_after,
            "unique_vessels": unique_vessels,
            "vessels_interpolated": len(interpolated_data),
            "file": file_path.name,
        }

        # Cleanup
        del df, interpolated_data
        gc.collect()

        return date_str, stats, detected_cols

    except Exception as e:
        return date_str, {"error": str(e)[:50]}, detected_cols


def process_daywise_data_simple(input_dir, checkpoint_dir):
    """
    Simple sequential processing: one file, complete, save, move to next.
    """

    checkpoint_dir.mkdir(exist_ok=True, parents=True)
    all_files = sorted(
        list(input_dir.glob("*.csv")) + list(input_dir.glob("*.parquet"))
    )

    # DEBUG: Show what was found
    print(f"\n[DEBUG] Input dir: {input_dir}")
    print(f"[DEBUG] Input dir exists: {input_dir.exists()}")
    print(f"[DEBUG] CSV files found: {len(list(input_dir.glob('*.csv')))}")
    print(f"[DEBUG] Parquet files found: {len(list(input_dir.glob('*.parquet')))}")
    print(f"[DEBUG] Total files to process: {len(all_files)}")

    print(f"\nProcessing {len(all_files)} files sequentially...\n")
    print("=" * 120)
    print(
        f"{'Date':<12} {'File':<40} {'Before':<12} {'After':<12} {'Vessels':<10} {'Saved':<10} {'Status':<30}"
    )
    print("=" * 120)

    stats_all = {}
    detected_cols = None

    for file_path in all_files:
        date_str, stats, detected_cols = process_single_file(
            file_path, checkpoint_dir, detected_cols
        )
        stats_all[date_str] = stats

        # Display result
        if "error" in stats:
            print(
                f"{date_str:<12} {file_path.name:<40} {'?':<12} {'?':<12} {'?':<10} {'?':<10} {'❌ ' + stats['error']:<30}"
            )
        else:
            status = f"✓ {stats['vessels_interpolated']} vessels"
            print(
                f"{date_str:<12} {file_path.name:<40} {stats['total_points_before']:<12} {stats['total_points_after']:<12} {stats['unique_vessels']:<10} {stats['vessels_interpolated']:<10} {status:<30}"
            )

    print("=" * 120)
    return stats_all


print("✓ Simple sequential processing function loaded (FIXED stats keys)")

✓ Simple sequential processing function loaded (FIXED stats keys)


In [6]:
# Simple sequential processing - no fancy features, just process files
print("=" * 100)
print("Starting SIMPLE Sequential AIS Preprocessing")
print("=" * 100)

try:
    stats = process_daywise_data_simple(processed_data_dir, checkpoint_dir)
    print("\n✓ Processing complete!")

except KeyboardInterrupt:
    print("\n⚠️  Interrupted by user")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback

    traceback.print_exc()

# Show summary
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

pkl_files = list(checkpoint_dir.glob("*.pkl"))
parquet_files = list(checkpoint_dir.glob("*.parquet"))

print(f"\nSaved files:")
print(f"  Pickle files: {len(pkl_files)}")
print(f"  Parquet files: {len(parquet_files)}")

if pkl_files:
    total_size = sum(f.stat().st_size for f in pkl_files + parquet_files) / (1024**2)
    print(f"  Total size: {total_size:.2f} MB")
    print(f"  Location: {checkpoint_dir.absolute()}")

Starting SIMPLE Sequential AIS Preprocessing

[DEBUG] Input dir: F:\PyTorch_GPU\maritime_monitoring_preprocessing\processed_data
[DEBUG] Input dir exists: True
[DEBUG] CSV files found: 8
[DEBUG] Parquet files found: 0
[DEBUG] Total files to process: 8

Processing 8 files sequentially...

Date         File                                     Before       After        Vessels    Saved      Status                        


  05:   0%|          | 0/13618 [00:00<?, ?v/s]

05           AIS_2020_01_05.csv                       6815545      0            13618      0          ✓ 0 vessels                   


  06:   0%|          | 0/14366 [00:00<?, ?v/s]


⚠️  Interrupted by user

SUMMARY

Saved files:
  Pickle files: 0
  Parquet files: 0


In [ ]:
# Generate and display statistics
if stats and len(stats) > 0:
    print("\n" + "=" * 100)
    print("PREPROCESSING STATISTICS SUMMARY")
    print("=" * 100 + "\n")

    # Filter out error entries
    valid_stats = {k: v for k, v in stats.items() if "error" not in v}

    if valid_stats:
        # Create DataFrame for better visualization
        stats_df = pd.DataFrame.from_dict(valid_stats, orient="index")
        stats_df = stats_df.sort_index()

        print("Per-Day Statistics:")
        print(stats_df.to_string())

        print("\n" + "=" * 100)
        print("OVERALL STATISTICS")
        print("=" * 100)

        total_days = len(valid_stats)
        total_vessels = stats_df["unique_vessels"].sum()
        total_points_before = stats_df["total_points_before"].sum()
        total_points_after = stats_df["total_points_after"].sum()

        if total_points_before > 0:
            avg_vessels_per_day = stats_df["unique_vessels"].mean()
            avg_points_per_day = stats_df["total_points_before"].mean()
            expansion_factor = total_points_after / total_points_before

            print(f"\n  Total Days Processed: {total_days}")
            print(f"  Total Unique Vessels (across all days): {int(total_vessels)}")
            print(f"  Average Vessels per Day: {avg_vessels_per_day:.1f}")
            print(
                f"\n  Total Data Points (Before Interpolation): {int(total_points_before):,}"
            )
            print(
                f"  Total Data Points (After Interpolation): {int(total_points_after):,}"
            )
            print(f"  Average Points per Day: {avg_points_per_day:.0f}")
            print(f"  Data Increase Factor: {expansion_factor:.2f}x")

            print("\n" + "=" * 100)
            print("SAVED FILES")
            print("=" * 100)

            pkl_files = list(checkpoint_dir.glob("*.pkl"))
            parquet_files = list(checkpoint_dir.glob("*.parquet"))

            print(f"\n  Pickle Files (.pkl): {len(pkl_files)}")
            for f in sorted(pkl_files):
                size_mb = f.stat().st_size / (1024**2)
                print(f"    - {f.name:<45} ({size_mb:>8.2f} MB)")

            print(f"\n  Parquet Files (.parquet): {len(parquet_files)}")
            for f in sorted(parquet_files):
                size_mb = f.stat().st_size / (1024**2)
                print(f"    - {f.name:<45} ({size_mb:>8.2f} MB)")

            total_size_mb = sum(f.stat().st_size for f in pkl_files + parquet_files) / (
                1024**2
            )
            print(f"\n  Total Storage Used: {total_size_mb:.2f} MB")
            print(f"  Output Directory: {checkpoint_dir.absolute()}")
        else:
            print("\n⚠️  No valid data points found after interpolation!")
    else:
        print("⚠️  All files had errors. Check error messages above.")
else:
    print("⚠️  No data was processed. Check input directory and file formats.")

In [ ]:
# Create visualizations from statistics
if stats and len(stats) > 0:
    # Filter valid stats
    valid_stats = {k: v for k, v in stats.items() if "error" not in v}

    if valid_stats:
        stats_df = pd.DataFrame.from_dict(valid_stats, orient="index").sort_index()

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(
            "AIS Data Interpolation Preprocessing Report",
            fontsize=16,
            fontweight="bold",
        )

        # 1. Unique Vessels per Day
        ax = axes[0, 0]
        dates = range(len(stats_df))
        ax.bar(
            dates,
            stats_df["unique_vessels"],
            color="steelblue",
            alpha=0.7,
            edgecolor="black",
        )
        ax.set_xlabel("Day", fontweight="bold")
        ax.set_ylabel("Number of Unique Vessels", fontweight="bold")
        ax.set_title("Unique Vessels per Day", fontweight="bold")
        ax.grid(axis="y", alpha=0.3)
        ax.set_xticks(dates)
        ax.set_xticklabels(stats_df.index, rotation=45)

        # 2. Data Points Before/After Interpolation
        ax = axes[0, 1]
        x = np.arange(len(stats_df))
        width = 0.35
        ax.bar(
            x - width / 2,
            stats_df["total_points_before"],
            width,
            label="Before",
            color="lightcoral",
            alpha=0.8,
        )
        ax.bar(
            x + width / 2,
            stats_df["total_points_after"],
            width,
            label="After",
            color="lightgreen",
            alpha=0.8,
        )
        ax.set_xlabel("Day", fontweight="bold")
        ax.set_ylabel("Number of Data Points", fontweight="bold")
        ax.set_title("Data Points: Before vs After Interpolation", fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(stats_df.index, rotation=45)
        ax.legend()
        ax.grid(axis="y", alpha=0.3)

        # 3. Interpolation Expansion Factor
        ax = axes[1, 0]
        expansion = stats_df["total_points_after"] / stats_df["total_points_before"]
        ax.plot(
            dates, expansion, marker="o", linewidth=2, markersize=8, color="darkblue"
        )
        ax.fill_between(dates, expansion, alpha=0.3, color="skyblue")
        ax.set_xlabel("Day", fontweight="bold")
        ax.set_ylabel("Expansion Factor", fontweight="bold")
        ax.set_title("Interpolation Expansion Factor (After/Before)", fontweight="bold")
        ax.grid(True, alpha=0.3)
        ax.set_xticks(dates)
        ax.set_xticklabels(stats_df.index, rotation=45)
        ax.axhline(
            y=expansion.mean(),
            color="red",
            linestyle="--",
            alpha=0.5,
            label=f"Mean: {expansion.mean():.2f}x",
        )
        ax.legend()

        # 4. Summary Statistics (Text)
        ax = axes[1, 1]
        ax.axis("off")

        total_days = len(stats_df)
        total_vessels = stats_df["unique_vessels"].sum()
        total_points_before = stats_df["total_points_before"].sum()
        total_points_after = stats_df["total_points_after"].sum()
        avg_vessels = stats_df["unique_vessels"].mean()
        overall_expansion = (
            total_points_after / total_points_before if total_points_before > 0 else 0
        )

        summary_text = f"""
PREPROCESSING SUMMARY

Total Days Processed: {total_days}
Total Unique Vessels: {int(total_vessels):,}
Average Vessels/Day: {avg_vessels:.1f}

Total Data Points (Before): {int(total_points_before):,}
Total Data Points (After): {int(total_points_after):,}
Overall Expansion Factor: {overall_expansion:.2f}x

Interpolation Method: Cubic Spline (5-min intervals)
Storage Format: Pickle + Parquet (Snappy)
Output Directory: {checkpoint_dir.name}
"""

        ax.text(
            0.1,
            0.95,
            summary_text,
            transform=ax.transAxes,
            fontsize=11,
            verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )

        plt.tight_layout()
        plt.show()

        print("\n✅ Visualization complete!")
    else:
        print("⚠️  No valid data to visualize. All entries contain errors.")
else:
    print("⚠️  No statistics available. Run the processing pipeline first.")

In [ ]:
# Utility functions for loading interpolated data from checkpoints
def load_interpolated_checkpoint(checkpoint_path, date_str=None, file_format="pkl"):
    """
    Load interpolated data from checkpoint files.

    Parameters:
    -----------
    checkpoint_path : Path or str
        Path to checkpoint directory
    date_str : str, optional
        Specific date to load (e.g., '2020_01_03'). If None, returns all files.
    file_format : str
        'pkl' for pickle files or 'parquet' for parquet files

    Returns:
    --------
    dict or pd.DataFrame
        Loaded data
    """
    checkpoint_path = Path(checkpoint_path)

    if date_str:
        if file_format == "pkl":
            file_path = checkpoint_path / f"interpolated_{date_str}.pkl"
            if file_path.exists():
                with open(file_path, "rb") as f:
                    return pickle.load(f)
        elif file_format == "parquet":
            file_path = checkpoint_path / f"interpolated_{date_str}.parquet"
            if file_path.exists():
                return pd.read_parquet(file_path)
        print(f"⚠️  File not found: {file_path}")
        return None
    else:
        # Load all files
        if file_format == "pkl":
            files = sorted(checkpoint_path.glob("*.pkl"))
            all_data = {}
            for f in files:
                with open(f, "rb") as fp:
                    data = pickle.load(fp)
                    all_data.update(data)
            return all_data
        elif file_format == "parquet":
            files = sorted(checkpoint_path.glob("*.parquet"))
            dfs = [pd.read_parquet(f) for f in files]
            return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def get_checkpoint_summary(checkpoint_path):
    """Get summary of all checkpoint files."""
    checkpoint_path = Path(checkpoint_path)

    pkl_files = sorted(checkpoint_path.glob("*.pkl"))
    parquet_files = sorted(checkpoint_path.glob("*.parquet"))

    print(f"\n{'='*80}")
    print(f"CHECKPOINT FILES SUMMARY")
    print(f"{'='*80}\n")

    print(f"Location: {checkpoint_path.absolute()}\n")

    print(f"Pickle Files ({len(pkl_files)}):")
    for f in pkl_files:
        size_mb = f.stat().st_size / (1024**2)
        print(f"  {f.name:<45} ({size_mb:>8.2f} MB)")

    print(f"\nParquet Files ({len(parquet_files)}):")
    for f in parquet_files:
        size_mb = f.stat().st_size / (1024**2)
        print(f"  {f.name:<45} ({size_mb:>8.2f} MB)")

    total_size = sum(f.stat().st_size for f in pkl_files + parquet_files) / (1024**2)
    print(f"\nTotal Storage: {total_size:.2f} MB")
    print(f"{'='*80}\n")


# Display checkpoint summary
get_checkpoint_summary(checkpoint_dir)